# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, process, and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nVersion: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all the record sets, their `@id`s, and the associated fields and field IDs for structured exploration. All entities are referenced by their `@id`.

In [ ]:
# Explore all record sets and their fields by @id
from pprint import pprint

record_sets = list(dataset.record_sets)
print(f"Number of record sets in dataset: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    print(f"Record Set: {rs['name']} (@id: {rs_id})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        field_id = field.get('@id', None) if isinstance(field, dict) else field
        if isinstance(field, dict):
            name = field.get('name', 'N/A')
        else:
            name = 'N/A'
        print(f"    - {name} (@id: {field_id})")
    print("")
pprint(record_set_ids)

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis. Here, we use each record set's `@id`.

In [ ]:
# Extract all record sets and records by @id
dfs = {}

# You may want to extract only a subset. Here, we extract all.
for rs_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        # Only create a DataFrame if there is at least one record
        if records:
            dfs[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(dfs[rs_id])} records for Record Set @id: {rs_id}")
            print(f"Fields: {dfs[rs_id].columns.tolist()}\n")
        else:
            print(f"No records found for Record Set @id: {rs_id}\n")
    except Exception as e:
        print(f"Could not load records for Record Set @id: {rs_id} due to error: {e}\n")

# For demonstration, pick the first non-empty DataFrame
main_rs_id = None
for rset, df in dfs.items():
    if not df.empty:
        main_rs_id = rset
        break
if main_rs_id:
    print(f"\nMain analysis will use Record Set @id: {main_rs_id}")
    print(dfs[main_rs_id].head())
else:
    print("No data available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, or grouping. Here, we select a numeric field for demonstration. Please adjust the field `@id`s based on your data structure.

In [ ]:
# Select a DataFrame for EDA (main_rs_id defined previously)
df = dfs[main_rs_id]
print(f"Columns in DataFrame: {df.columns.tolist()}")

# Try to identify numeric fields. If not evident, set one by @id.
numeric_candidate = None
for col in df.columns:
    # Try selecting first float/int-like column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_candidate = col
        break
if not numeric_candidate:
    # Try automatic conversion of suitable columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_candidate = col
                break
        except:
            continue

if numeric_candidate:
    numeric_field_id = numeric_candidate
    print(f"Numeric field selected for analysis: {numeric_field_id}")
    threshold = df[numeric_field_id].quantile(0.5)  # Example: median

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to select a grouping field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(grouped_df.head())
else:
    print("No numeric field found in DataFrame. Please check the dataset fields.")

## 5. Visualization
Visualize the data distribution and relationships using Pandas/Matplotlib/Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If EDA step found a numeric field, show its distribution
if 'numeric_field_id' in locals():
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Example: Scatter plot with numeric vs category, if possible
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} across {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field present for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² colorectal cancer survivors dataset via its Croissant schema, explored its record sets and fields, performed example data extraction and elementary EDA operations, and visualized some of the field distributions using Python data science tools.

Key steps:
- **All record sets, fields, and columns were referenced strictly by their `@id`.**
- Demonstrated dynamic identification of numeric fields and group fields for adaptable analysis.
- Provided templates for filtering, normalization, grouping, and visualization to jumpstart further research.

For more advanced analysis, consider integrating clinical or molecular attribute interpretation, cross-record set joins (using shared `@id` references), and model training. Consult the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) documentation for further functionality.